# 🏥 ask.39.net High-Throughput Distributed Crawler (Shard Worker)

**Dự án ViBioMIR 2026** - Cào kho hỏi đáp lâm sàng Bác sĩ - Bệnh nhân (337.375 URLs).

### Hướng dẫn 1-Click:
1. Chạy **Cell 1** để cài đặt Patchright và toàn bộ thư viện hệ thống Linux (`--with-deps`).
2. Chọn **SHARD_NUM** tương ứng từ `1` đến `6` ở **Cell 2** và bấm chạy.
3. Khi hoàn tất, tải file `ask39_shard_{SHARD_NUM}.jsonl` hoặc `.sqlite` về máy.

In [ ]:
# Step 1: Install Patchright Chromium Engine & System Shared Libraries
!pip install -q patchright beautifulsoup4 pyarrow
!patchright install --with-deps chromium

# Sanity Check: Verify Chromium engine launches smoothly in Jupyter async loop
from patchright.async_api import async_playwright
async with async_playwright() as pw:
    b = await pw.chromium.launch(headless=True, args=["--no-sandbox"])
    await b.close()
print("✅ Trình duyệt Chromium đã sẵn sàng 100%, không còn lỗi thư viện!")


In [ ]:
# Step 2: Configure Shard Number (CHỈNH SỐ SHARD TẠI ĐÂY: 1, 2, 3, 4, 5, 6)
SHARD_NUM = 1  # <-- Đổi số này tương ứng cho từng notebook (từ 1 đến 6)
CONCURRENCY = 4
BATCH_SIZE = 25

import os, glob

# Locate worker script
worker_paths = glob.glob('/kaggle/input/**/ask39_kaggle_worker.py', recursive=True) + ['ask39_kaggle_worker.py']
worker_script = worker_paths[0] if worker_paths and os.path.exists(worker_paths[0]) else 'ask39_kaggle_worker.py'

# Locate shard parquet file
parquet_paths = glob.glob(f'/kaggle/input/**/ask39_shard_{SHARD_NUM}.parquet', recursive=True)
parquet_arg = f'--parquet {parquet_paths[0]}' if parquet_paths else ''

print(f'Starting Shard {SHARD_NUM} with worker: {worker_script} ...')
!python {worker_script} --shard {SHARD_NUM} {parquet_arg} --concurrency {CONCURRENCY} --batch-size {BATCH_SIZE}

In [ ]:
# Step 3: Verify Output Integrity
!ls -lh /kaggle/working/ask39_shard_*
!wc -l /kaggle/working/ask39_shard_*.jsonl
print('\n--- Mẫu 2 dòng dữ liệu đầu tiên trích xuất ---')
!head -n 2 /kaggle/working/ask39_shard_*.jsonl